# ML-05 — Feature Vector and Leakage/Privacy Check

This notebook builds the feature vector and verifies that no future-looking or label-derived variables are leaked into the training set. It also ensures client privacy by keeping hashed IDs out of model features.

## 1. Build the feature vector

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.append(str(Path('../../scripts')))
from ml_utils import RAW_PATH, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

# Load processed features
features_path = Path('../../../data/processed/refresh_feature_vector.csv')
if not features_path.exists():
    # Run preparation if missing
    import subprocess
    subprocess.run([sys.executable, '../../scripts/01_prepare_features.py'])
df = pd.read_csv(features_path)
print(f"Loaded feature vector shape: {df.shape}")
print("Model numeric features:", MODEL_NUMERIC_FEATURES)
print("Model categorical features:", MODEL_CATEGORICAL_FEATURES)

## 2. Feature notes (meaning, missing, categorical, available-when?)

- `search_volume`: Trailing search interest volume, knowable before splitting.
- `word_count`: Length of page content, knowable prior to prediction.
- `days_since_last_update`: Days since the content was last refreshed. Knowable from CMS metadata.
- `ctr`: Click-through rate in GSC. Derived from historic baseline window.
- `avg_position`: Mean ranking position in search results. Knowable before split moment.

All missing numeric values are filled with `0` or appropriate medians, and categorical ones with `'unknown'`.

## 3. The leakage hunt

We verify that the target variable `is_declining_label` and any label-derived fields (`trend_direction`, `trend_pct`) are NOT in the training feature columns.

In [ ]:
features_to_check = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
leakage_cols = ['trend_direction', 'trend_pct', 'is_declining_label', 'imp_future15', 'clk_future15']
leaked = [col for col in leakage_cols if col in features_to_check]
print(f"Leaked columns in features: {leaked}")
assert len(leaked) == 0, "Leakage detected!"

# Verify target correlation
numeric_df = df[MODEL_NUMERIC_FEATURES].copy()
numeric_df['label'] = df['is_declining_label']
corrs = numeric_df.corr()['label'].abs().sort_values(ascending=False)
print("Feature correlation with label (none should be 1.0 except the label itself):")
print(corrs.head(10))

## 4. What I excluded and why

- `trend_direction` / `trend_pct`: Directly summarizes future outcome period, leading to 100% target leakage.
- `client_id` / `content_id`: Used for grouping/splitting only; if used as features, they lead to client-specific overfitting and poor generalization to new clients.